# Data Quality Audit — 07 · Entertainment in Saudi Arabia (Kaggle)

**Source:** `data/raw/entertainment/Entertainment_KSA.csv`

**Purpose in the concierge:** Entertainment venues: ratings, categories, locations.

Standardized audit covering:

```
Dataset
├── Shape
├── Columns & data types
├── Missing values
├── Duplicates
├── Invalid values
├── Outliers
├── Inconsistent categories
├── Geographic validity
├── Date/time validity
├── Data-source/license
└── Known limitations
```

> This is a scraped, Google-Maps-style file and the **messiest** tabular source (column misalignment, non-Saudi rows).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

# Resolve repo root whether run from the audit folder or the repo root.
p = Path.cwd()
while p != p.parent and not (p / "data" / "raw").exists():
    p = p.parent
ROOT = p
print("repo root:", ROOT)

# Saudi Arabia bounding box (approx) for geographic validity checks.
SA_LAT = (16.0, 32.5)
SA_LON = (34.5, 56.0)

def iqr_outliers(series):
    """Return (count, lower, upper) of IQR outliers in a numeric series."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return 0, np.nan, np.nan
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < lo) | (s > hi)).sum()), round(lo, 2), round(hi, 2)

def missing_report(df):
    m = pd.DataFrame({"missing": df.isna().sum(),
                      "missing_%": (df.isna().mean() * 100).round(1)})
    return m[m["missing"] > 0].sort_values("missing", ascending=False)

def dtype_report(df):
    return pd.DataFrame({
        "dtype": [str(t) for t in df.dtypes],
        "non_null": df.notna().sum().values,
        "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
    }, index=df.columns)

import re
df = pd.read_csv(ROOT / "data/raw/entertainment/Entertainment_KSA.csv").rename(columns={"Unnamed: 0":"idx"})
print("Loaded entertainment:", df.shape)
df.head(3)

repo root: /home/user/saudi-Digital-Concierge
Loaded entertainment: (564, 7)


,idx,name,rating,review_count,genre,location,best_comment
0,0,المسرح الروماني | Roman Theater,4.1,(151),Performing arts theater,Al Uqayr Saudi Arabia,NaN
1,1,Maraya,4.6,(1.1T),Concert hall,Al Atheeb Saudi Arabia,"""Also the theater is another level of perfecti..."
2,2,"Novo Cinemas, Seef Mall",4.3,(801),Movie theater,"Muharraq, Bahrain · In Seef Mall - Arad, Muharraq","""The theater was comfortable and the screening..."


## Shape

In [2]:
print("Rows:", len(df), "| Columns:", df.shape[1])

Rows: 564 | Columns: 7


## Columns & data types
`rating` and `review_count` are stored as text (should be numeric).

In [3]:
dtype_report(df)

,dtype,non_null,n_unique
idx,int64,564,564
name,str,559,271
rating,str,564,31
review_count,str,562,211
genre,str,541,50
location,str,539,120
best_comment,str,342,159


## Missing values

In [4]:
missing_report(df.drop(columns='idx'))

,missing,missing_%
best_comment,222,39.4
location,25,4.4
genre,23,4.1
name,5,0.9
review_count,2,0.4


## Duplicates

In [5]:
print("Exact duplicate rows:", df.drop(columns="idx").duplicated().sum())
print("Unique names:", df["name"].nunique(), "/", len(df))

Exact duplicate rows: 243
Unique names: 271 / 564


## Invalid values — column misalignment
~24 rows are shifted: `rating` holds text (`No reviews · …`, `Saudi Arabia`) and `genre` holds price symbols (`₹`).

In [6]:
rating_num = pd.to_numeric(df["rating"], errors="coerce")
print("Non-numeric rating rows (misaligned/no-reviews):", int(rating_num.isna().sum()))
print("rating out of 0-5:", int(((rating_num<0)|(rating_num>5)).sum()))
print("\nExamples of bad rating values:")
print(df.loc[rating_num.isna(),"rating"].value_counts().head(8).to_string())
g = df["genre"].fillna("").str.strip()
print("\nContaminated genre (price symbols / location leaks):", int((g.str.match(r"^[₹$]+$") | g.str.startswith("In ")).sum()))

Non-numeric rating rows (misaligned/no-reviews): 24
rating out of 0-5: 0

Examples of bad rating values:
rating
No reviews · Tourist attraction    13
No reviews · Movie theater          2
No reviews · Amphitheater           2
No reviews · IMAX theater           2
Saudi Arabia                        2
No reviews · Drama theater          1
No reviews · Restaurant             1
No reviews · School                 1

Contaminated genre (price symbols / location leaks): 18


## Outliers
Parse `review_count` (`(1.1T)` → 1,100) and report outliers.

In [7]:
def parse_rc(x):
    if pd.isna(x): return np.nan
    m = re.match(r"([\d.]+)\s*([KkMmTt]?)", str(x).strip().strip("()").strip())
    if not m: return np.nan
    return float(m.group(1)) * {"":1,"K":1e3,"T":1e3,"M":1e6}[m.group(2).upper()]
rc = df["review_count"].map(parse_rc)
n, lo, hi = iqr_outliers(rc)
print(f"review_count parse fails: {int(rc.isna().sum())} | IQR outliers: {n} (bounds {lo}..{hi})")
print("rating stats:", pd.to_numeric(df["rating"],errors="coerce").describe().round(2).to_dict())

review_count parse fails: 24 | IQR outliers: 55 (bounds -2368.12..4140.88)
rating stats: {'count': 540.0, 'mean': 4.21, 'std': 0.39, 'min': 2.5, '25%': 4.0, '50%': 4.2, '75%': 4.5, 'max': 5.0}


## Inconsistent categories
`genre` has leading spaces + contamination; names repeat (chains) with/without the Arabic half.

In [8]:
gc = df["genre"].fillna("").str.strip()
contam = gc.str.match(r"^[₹$]+$") | gc.str.startswith("In ")
print(gc[~contam & (gc!="")].value_counts().head(15).to_string())

genre
Tourist attraction             132
Movie theater                  128
Amusement center                57
Park                            23
Children's amusement center     21
Escape room center              20
Performing arts theater         16
Theme park                      13
Amusement park                  10
Garden                          10
Hiking area                      8
Concert hall                     7
Stage                            6
State park                       5
Museum                           4


## Geographic validity
No coordinates; `location` is free text and includes **non-Saudi** venues (Bahrain, Kuwait).

In [9]:
loc = df["location"].fillna("")
country = np.select([loc.str.contains("Saudi Arabia"), loc.str.contains("Bahrain"),
                     loc.str.contains("Kuwait"), loc.eq("")],
                    ["Saudi Arabia","Bahrain","Kuwait","Unknown"], default="Other")
print(pd.Series(country).value_counts().to_dict())

{'Saudi Arabia': 525, 'Unknown': 25, 'Bahrain': 7, 'Other': 7}


## Date/time validity
No date/time columns — **N/A**.

In [10]:
print("Date-like columns:", [c for c in df.columns if any(k in c.lower() for k in ["date","time","year"])])

Date-like columns: []


## Data-source / license
- **Source:** Kaggle — *Entertainment in Saudi Arabia*.
- **License:** TBD.
- **Currency:** static snapshot.

## Known limitations
- **Column misalignment (~24 rows)** — the main issue; detect via non-numeric `rating`.
- **`rating` / `review_count` are text** → parse to numeric.
- **~14 non-Saudi venues** (Bahrain/Kuwait) despite the KSA name → filter.
- **`genre` contaminated** (price symbols, location leaks) + leading spaces.
- **`best_comment` ~39% missing**; names inconsistent (with/without Arabic).
- No coordinates; `location` is free text.